In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as snsz
from sklearn.linear_model import LinearRegression
import scipy.stats as stats
from sklearn.metrics import r2_score
from sklearn.naive_bayes import MultinomialNB, GaussianNB, CategoricalNB
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler

# SEMANA 3
#### PASO 1: 
En  esta  semana  se  usará  otro  tipo  de  modelo,  el  Naïve  Bayes,  y  se  realizará  una 
implementación propia. Para ello escribe la clase NaïveBayes, implementando al menos 
los métodos siguientes: 
 
  
El  atributo  Laplace  indica  si  debe  o  no  aplicarse  la  corrección  de  Laplace.  Este  y 
cualquier  otro  atributo  que  necesites  debe  pasarse  al  constructor  de  la  clase.  Los 
métodos  fit,  predict  y  score  implementan  una  funcionalidad  similar  a  la  que  hemos 
comprobado en las librerías de Scikit Learn: 
 
• fit(X,y):  Calcula  el  modelo  correspondiente  a  partir  de  los  datos  de  train.  En 
nuestra implementación se puede pasar un data frame que analice los atributos 
del  dataset  y  calcule  las  probabilidades  teniendo  en  cuenta  si  son  valores 
nominales (tipo object) o numéricos (tipo int, float...). En Naïve Bayes el modelo 
es el conjunto de verosimilitudes y de probabilidades a priori, calculadas a partir 
de los datos de train. 
• predict(X): devuelve el conjunto de clases estimadas para cada uno de los datos 
de test 
• score(X, y): devuelve el accuracy medio, es decir, los aciertos en las 
predicciones del conjunto pasado como argumento.

In [52]:
class NaiveBayes:
  
    # Constructor: pasar argumentos necesarios
  def __init__(self, laplace=False):   
      self.laplace = laplace
      # Las clases (se rellenan en fit)
      self.clases = None
      
      # Probabilidades previas de las clases P(y)
      self.prior = None
      
      # Diccionario para guardar si cada atributo es numérico o categórico
      # (es importante tratarlos de forma distinta por cómo se modelan sus verosimilitudes)
      self.tipo = {}
        
      # Aquí guardaremos parámetros como medias, desviaciones, frecuencias, etc.:
      self.fit_parameters = {}
      
      
      # Probabilidades condicionadas de cada atributo dado cada clase P(x_j | y):
      # Esto permitirá evaluar nuevas muestras sin recalcular desde cero.
      self.probabilities = {}
  # Calcula el modelo correspondiente a partir de los datos de train
  def fit(self, X, y):
    #Identificar las clases
    # Nos interesa conocer explícitamente todos los valores posibles de la variable objetivo
    # porque el modelo construye una distribución de probabilidades por cada una de ellas.
    self.clases = sorted(pd.Series(y).unique())

    # Calcular las probabilidades a priori P(c)
    # La proporción de muestras de cada clase actúa como punto de partida de las predicciones.
    self.prior = {}
    y_series = pd.Series(y)
    total_muestras = len(y_series)
    for i in self.clases:
        n_i = (y_series == i).sum()
        self.prior[i] = n_i / total_muestras

    
    # Determinar el tipo de cada atributo
    # Distinguir entre continuos y nominales
    # ya que la forma de modelar P(A_i | c) depende del tipo de dato.
    self.tipo = {}

    for col in X.columns:

        # Columnas como IDs no aportan información útil al modelo
        if col == "Customer ID":
            continue

        # Para los continuos usaremos una distribución normal
        if np.issubdtype(X[col].dtype, np.number):
            self.tipo[col] = "num"
        else:
            # Para los nominales trabajaremos con frecuencias normalizadas
            self.tipo[col] = "cat"

    # Preparar la estructura donde almacenaremos los parámetros del modelo
    self.fit_parameters = {c: {} for c in self.clases}


    # Las probabilidades condicionadas de atributos nominales también se organizan por clase.
    self.probabilities = {c: {} for c in self.clases}




    # Atributos numéricos (distribución normal)
    # Para cada clase y atributo continuo, obtenemos media y desviación,
    # lo cual permite capturar la variabilidad del dato
    for i in self.clases:
        X_i = X[y == i]

        for col in self.tipo:
            if self.tipo[col] == "num":

                mu = X_i[col].mean()
                sigma = X_i[col].std()

                # Evitar desviaciones extremadamente bajas que darían problemas.
                if sigma < 1e-6:
                    sigma = 1e-6

                self.fit_parameters[i][col] = (mu, sigma)





    # Atributos categóricos (frecuencias normalizadas)
    # Se cuentan los valores por clase y se normalizan.
    # Cuando alguna combinación aparece poco, el suavizado opcional evita que
    # la probabilidad se anule completamente.
    for c in self.clases:
        X_c = X[y == c]

        for col in self.tipo:
            if self.tipo[col] == "cat":

                # Conteos de cada categoría dentro de la clase c
                counts = X_c[col].value_counts()
                total_c = counts.sum()
                K = len(counts)  # nº categorías diferentes observadas

                self.probabilities[c][col] = {}

                for valor, freq in counts.items():

                    if self.laplace:
                        # Suavizado aditivo para evitar probabilidades nulas
                        prob = (freq + 1) / (total_c + K)
                    else:
                        prob = freq / total_c

                    self.probabilities[c][col][valor] = prob




    return self
  





# Probamos inicialmente a trabajar con productos directos de probabilidades,
# pero al combinar muchas verosimilitudes pequeñas se acababan convirtiendo en 0
# debido a la precisión limitada de python. Para evitar este efecto,
# usamos logaritmos, donde las multiplicaciones pasan a sumas y se mantiene
# estabilidad numérica incluso con valores muy pequeños.
# Ahora haremos una funcion llamada predict_proba que devuelva las probabilidades logarítmicas
# de cada clase para cada muestra de entrada X. Esta devolverá una matriz de tamaño
# (n_filas, n_clases) donde cada fila contiene los logaritmos de
# las probabilidades de pertenencia a cada clase.



  def predict_proba(self, X):

    resultados = []
    for _, fila in X.iterrows():
        log_probs = {}

        # Evaluamos cada clase posible
        for c in self.clases:

            # Comenzamos por el término log(P(c))
            lp = np.log(self.prior[c])

            # Recorremos todos los atributos modelados
            for col in self.tipo:

                valor = fila[col]

                # Atributo numérico: distribución normal

                if self.tipo[col] == "num":
                    mu, sigma = self.fit_parameters[c][col]

                    # Densidad de la normal en forma logarítmica por lo que dijimos antes
                    lp += - np.log(np.sqrt(2 * np.pi) * sigma) / - ((valor - mu) ** 2) / (2 * sigma ** 2)

                # Atributo categórico
                else:
                    probs_col = self.probabilities[c][col]

                    if valor in probs_col:
                        # Categoría vista en entrenamiento
                        lp += np.log(probs_col[valor])
                    else:
                        # Categoría no vista: aplicamos suavizado si procede
                        if self.laplace:
                            K = len(probs_col)
                            # Prob mínima coherente con el ajuste categórico
                            prob_min = 1 / (sum(probs_col.values()) + K)
                            lp += np.log(prob_min)
                        else:
                            # Sin suavizado, la verosimilitud colapsa a cero
                            lp += -np.inf

            log_probs[c] = lp

        # Convertimos log-probabilidades a probabilidades normales
        # usando "log-sum-exp" para mayor estabilidad
        max_log = max(log_probs.values())
        # Diccionario donde iremos guardando los valores exponenciados
        exp_vals = {}
        for c in self.clases:
          # Obtenemos el log-prob correspondiente a esa clase
          log_val = log_probs[c]

          # Restamos el máximo para estabilizar numéricamente
          log_val_adjusted = log_val - max_log

          # Exponencia para salir de los logaritmos
          exp_val = np.exp(log_val_adjusted)

          # Guardamos el valor en el diccionario
          exp_vals[c] = exp_val
        total = sum(exp_vals.values())

        probs_finales = [exp_vals[c] / total for c in self.clases]

        resultados.append(probs_finales)

    return np.array(resultados)




    
  # Ahora que tenemos las probabilidades, podemos implementar predict usando predict_proba.
  # Hemos implementado predict_proba que devuelve las probabilidades de cada clase para cada muestra
  # para que podamos meterlo en un for y implementar predict de forma sencilla.
  def predict(self,X):
    probabilidades = self.predict_proba(X)
     # Lista donde guardaremos las predicciones finales
    predicciones = []
    for fila in probabilidades:
        # Buscamos el índice de la probabilidad máxima de esta fila
        indice_max = np.argmax(fila)
        # Convertimos ese índice a la clase real correspondiente
        clase_predicha = self.clases[indice_max]
        predicciones.append(clase_predicha)

    # Convertimos a array por comodidad
    return np.array(predicciones)




  # devuelve el accuracy medio, es decir, los aciertos en las predicciones del conjunto pasado como argumento
  # Ahora que tenemos predict implementado, vamos a implementar score que nos servirá para evaluar el modelo.
  # Básicamente compara las predicciones con las reales.
  def score(self,X,y):
    predicciones = self.predict(X)
    aciertos = 0

    # 3. Recorremos una a una las filas comparando con la etiqueta real
    for pred, real in zip(predicciones, y):

        # Si la predicción coincide con el valor real, contamos un acierto
        if pred == real:
            aciertos += 1
    total = len(y)
    accuracy = aciertos / total

    return accuracy

#### PASO 2:
Utiliza  la  clase  Naïve  Bayes  creada  en  el  paso  anterior  para  calcular  el  score  del 
clasificador en el dataset con el que estamos trabajando en la práctica.

In [53]:
# Cargar los datos de construcción (train)
df = pd.read_csv("fuga_clientes_empresa_telefonica_construccion.csv")

# Separar X e y
# 'Churn Status' es la variable objetivo
y = df["Churn Status"]
X = df.drop(columns=["Churn Status"])

# Creamos una instancia del clasificador
# Activamos Laplace ya que el dataset contiene atributos categóricos 
modelo_nb = NaiveBayes(laplace=True)

# Ajustamos el modelo
modelo_nb.fit(X, y)

# Calculamos el score (accuracy)
score_nb = modelo_nb.score(X, y)

print("Accuracy del Naive Bayes propio en dataset de construcción:", score_nb)

Accuracy del Naive Bayes propio en dataset de construcción: 0.6678667866786678


##### PASO 3:
Clasifica  el  conjunto  de  datos  con  cada  uno  de  los  tres  algoritmos  y  compara  sus 
accuracy  con  el  obtenido  con  la  implementación  propia.  ¿Cuál  funciona  mejor?  ¿Por 
qué?

Gaussian

In [54]:
X_num = df.select_dtypes(include=[np.number]).drop(columns=["Churn Status"])

gnb = GaussianNB()
gnb.fit(X_num, y)
acc_gnb = gnb.score(X_num, y)
print("Accuracy del GaussianNB:", acc_gnb)

Accuracy del GaussianNB: 0.630963096309631


Multinomial

In [56]:
# MultinomialNB necesita variables no negativas enteras.
# Por lo tanto, hay que hacer onehot para las categoricas.
# Aun asi, siguen habiendo variables numéricas que pueden ser negativas o no enteras,
# por lo que hay que seleccionar solo las categóricas.

# Seleccionamos solo las columnas categóricas (excluimos numéricas y el ID)
X_cat = df.select_dtypes(exclude=[np.number]).drop(columns=["Customer ID"])

# One-hot encoding para convertir categorías en variables 0/1
encoder = OneHotEncoder(handle_unknown="ignore")
X_cat_encoded = encoder.fit_transform(X_cat).toarray()

# MultinomialNB está pensado para trabajar con este tipo de matriz de cuentas (no negativas)
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_cat_encoded, y)
acc_mnb = mnb.score(X_cat_encoded, y)

print("Accuracy MultinomialNB (solo categóricas one-hot):", acc_mnb)

Accuracy MultinomialNB (solo categóricas one-hot): 0.6831683168316832


Categorical

In [58]:
# Seleccionamos solo categóricas
X_cat = df.select_dtypes(exclude=[np.number]).drop(columns=["Customer ID"])

# NORMALIZACIÓN PROFUNDA DE CATEGORÍAS
X_cat_clean = X_cat.copy()

for col in X_cat_clean.columns:
    # Convertimos todo a string
    X_cat_clean[col] = X_cat_clean[col].astype(str)

    # Quitamos espacios
    X_cat_clean[col] = X_cat_clean[col].str.strip()

    # Reemplazamos vacíos, "nan", "None", "NaN", que sino factorize reempleza por -1 y CategoricalNB no lo acepta
    X_cat_clean[col] = X_cat_clean[col].replace(
        ["", "nan", "NaN", "None", "NONE", " "],
        "MISSING_VALUE"
    )

# Ahora factorizar NUNCA generará -1
X_cat_factorized = pd.DataFrame()
for col in X_cat_clean.columns:
    codes, uniques = pd.factorize(X_cat_clean[col], sort=True)
    X_cat_factorized[col] = codes
# Convertimos a numpy
X_cnb = X_cat_factorized.values
# Entrenar CategoricalNB
cnb = CategoricalNB()
cnb.fit(X_cnb, y)
acc_cnb = cnb.score(X_cnb, y)

print("Accuracy CategoricalNB:", acc_cnb)


Accuracy CategoricalNB: 0.6831683168316832


Después de implementar nuestro propio clasificador Naive Bayes, comparamos su rendimiento con las tres variantes disponibles en scikit-learn. El objetivo es analizar cuál de los enfoques se adapta mejor al tipo de datos del problema de fuga de clientes, teniendo en cuenta que el dataset combina variables numéricas y variables categóricas con distinta naturaleza y distribución.

Los resultados muestran que CategoricalNB y MultinomialNB alcanzan el mejor rendimiento, con un accuracy aproximado de 0.68 en ambos casos. Este comportamiento es razonable porque estas dos variantes trabajan con las variables categóricas del dataset, que en este problema contienen información relevante sobre el comportamiento del cliente y suelen estar bien representadas como categorías discretas. Por el contrario, GaussianNB obtiene un rendimiento algo más bajo, cercano al 0.63, ya que supone que todas las características siguen distribuciones normales, una hipótesis que no se cumple en este dataset, especialmente para las variables categóricas y para algunas numéricas con distribuciones alejadas de gauss

La implementación propia obtiene un resultado cercano al 0.66, quedando entre GaussianNB y los enfoques categóricos. Esto se debe a que combina gaussianas para atributos numéricos y frecuencias para los categóricos, lo cual es razonable, pero sigue dependiendo de la independencia condicional entre todos los atributos. En un dataset con variables numéricas fuertemente correlacionadas, esta limitación hace que el rendimiento no llegue al de los modelos que explotan más directamente la estructura categórica. En conjunto, la variante categórica y la multinomial parecen adaptarse mejor a la naturaleza del conjunto de datos utilizado.
